# 1. Train the attendance model on Guest Talks: Higgsfield Edition

Registrations with a check-in label (`CheckIn`) from the Higgsfield event (18 Oct 2025). The name column is dropped before any analysis. Features: education degree, university, major group, how early the person registered, age, participation type and past AIESEC projects.

In [ ]:
import pandas as pd

# Load your dataset
df = pd.read_csv('data/Higgsfield_Registrations.csv')  # private registration export, not included
df

cols_to_fix = ['AIESEC Recruitment', 'Creative One', 'GO Exchanges', 'SheCodes', 'OZGE Forum', 'EmployMe', 'Other', 'Did not participate' ]
df[cols_to_fix] = df[cols_to_fix].notna()
df

# Create a new dataframe without the NaN names
df_cleaned = df.dropna(subset=['Your name and surname:'])
df_cleaned = df_cleaned.dropna(subset=['Your date of birth:'])
df_cleaned

df_cleaned.info()

col_name = 'Did you attend the Guest Talks: Higgsfield Edition?'
df_cleaned[col_name] = df_cleaned[col_name].astype(bool)

# Drops the name column
df_cleaned = df_cleaned.drop(columns=['Your name and surname:'])

df_cleaned

#Analysing average time to submit type form
# 1. Convert columns to datetime
df_cleaned['Submit Date (UTC)'] = pd.to_datetime(df_cleaned['Submit Date (UTC)'])
df_cleaned['Start Date (UTC)'] = pd.to_datetime(df_cleaned['Start Date (UTC)'])

# 2. Calculate the difference (Now it works!)
df_cleaned['time_typeform'] = df_cleaned['Submit Date (UTC)'] - df_cleaned['Start Date (UTC)']

# Optional: Convert the result to total minutes or seconds for CatBoost
# CatBoost works better with numbers than 'timedelta' objects
df_cleaned['time_taken_minutes'] = df_cleaned['time_typeform'].dt.total_seconds() / 60
df_cleaned_drop = df_cleaned[df_cleaned['time_taken_minutes'] < 10 ]

average_time_to_fill = df_cleaned_drop['time_taken_minutes'].mean()
print(average_time_to_fill)
df_cleaned_drop

from datetime import datetime
# 1. Set the event date
event_date = datetime(2025, 10, 18)

# 2. Convert Date of Birth to datetime and remove timezone info
df_cleaned['Your date of birth:'] = pd.to_datetime(df_cleaned['Your date of birth:'], errors='coerce').dt.tz_localize(None)

# 3. Calculate Age as a float
# We calculate total days difference and divide by 365.25
df_cleaned['Age'] = (event_date - df_cleaned['Your date of birth:']).dt.days / 365.25

# 4. Round to one or two decimal places
df_cleaned['Age'] = df_cleaned['Age'].round(2)
df_cleaned

# Standardize Major and Place of Study
df_cleaned['Your major:'] = df_cleaned['Your major:'].astype(str).str.lower().str.strip()
df_cleaned['Your place of study:'] = df_cleaned['Your place of study:'].astype(str).str.lower().str.strip()

df_cleaned

print(df_cleaned.nunique())

import pandas as pd

# 1. Set the event date
event_date = pd.Timestamp('2025-10-18').normalize()

# 2. Convert Submit Date to datetime and remove timezone info
df_cleaned['Submit Date (UTC)'] = pd.to_datetime(df_cleaned['Submit Date (UTC)']).dt.tz_localize(None)

# 3. Calculate the difference in full days
df_cleaned['days_before_event'] = (event_date - df_cleaned['Submit Date (UTC)'].dt.normalize()).dt.days

# 4. Define the category logic
def get_reg_category(days):
    if days <= 1:
        return "Less than day"
    else:
        # e.g., if 2 days before (Oct 16), it becomes "Day 1"
        return f"Day {days - 1}"

df_cleaned['Reg_Day_Category'] = df_cleaned['days_before_event'].apply(get_reg_category)

df_cleaned

def sort_key(cat):
    if cat == "Less than day":
        return 0
    return int(cat.split(" ")[1])
sorted_categories = sorted(df_cleaned['Reg_Day_Category'].unique(), key=sort_key)
ordered_counts = df_cleaned['Reg_Day_Category'].value_counts().reindex(sorted_categories)

print("Registration Day Categories and Counts:")
print(ordered_counts)

# CatBoost is listed in requirements.txt

def group_major(major):
    major = str(major).lower().strip()
    
    # Data Science & AI
    if any(x in major for x in ['data science', 'big data', 'artificial intelligence', 'ai', 'analytics']):
        return 'Data Science & AI'
    
    # Computer Science (CS/Software but not Electrical)
    if (('computer' in major or 'software' in major or major == 'cs') and 'electrical' not in major):
        return 'Computer Science'
    
    # IT & Cybersecurity
    if any(x in major for x in ['cybersecurity', 'cyber security', ' it', 'it ', 'information technology', 'systems', 'smart tech']):
        return 'IT & Cybersecurity'
    
    # Engineering
    if any(x in major for x in ['engineering', 'engineer', 'robotics', 'mae', 'civil', 'petroleum']):
        return 'Engineering'
    
    # Economics & Finance
    if any(x in major for x in ['economics', 'finance', 'accounting', 'econ']):
        return 'Economics & Finance'
    
    # Business & Management
    if any(x in major for x in ['management', 'marketing', 'business', 'mba']):
        return 'Business & Management'
    
    # Social Sciences & Humanities
    if any(x in major for x in ['political', 'relation', 'sociology', 'psir', 'journalism', 'literature', 'translation', 'psychology']):
        return 'Social Sciences & Humanities'
    
    # Natural Sciences & Math
    if any(x in major for x in ['math', 'biological', 'chemistry', 'biology', 'physics']):
        return 'Math & Natural Sciences'
    
    return 'Other'

# Apply the categorization
df_cleaned['Major_Category'] = df['Your major:'].apply(group_major)
df_cleaned
counts = df_cleaned['Major_Category'].value_counts()

print(counts)

import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split



# 2. Select your Features (X) and Target (y)
# Drop IDs, raw dates, and the target itself
X = df_cleaned.drop(columns=[
    'Your date of birth:', 
    'Start Date (UTC)', 'Submit Date (UTC)', 
    'time_typeform', 'CheckIn',
    'time_taken_minutes',
    'days_before_event', 'Your major:'
])
X

y = df_cleaned['CheckIn']

# 3. Define Categorical Columns (VERY IMPORTANT)
# These are the text-based columns
cat_features = ['Indicate whether you are participating as a team captain, a member of an existing team, or as an individual participant.', 
                'Your education degree:', 'Your place of study:', 'Major_Category', 'Reg_Day_Category']

# 4. Split and Train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    cat_features=cat_features, # This is the "magic" of CatBoost
    eval_metric='AUC',
    verbose=100
)

model.fit(X_train, y_train, eval_set=(X_test, y_test))

In [ ]:
# Get feature importance
importances = model.get_feature_importance(type="FeatureImportance")
feature_names = X.columns

# Display them in a nice list
print("--- FEATURE IMPORTANCE ---")
for score, name in sorted(zip(importances, feature_names), reverse=True):
    print(f'{name}: {score:.2f}%')

In [ ]:
import matplotlib.pyplot as plt

# Get feature importance from your trained model
importances = model.get_feature_importance(type="FeatureImportance")
feature_names = X.columns

# Create a DataFrame for easy plotting
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=True)

# Plot
plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='skyblue')
plt.xlabel('Importance Score (%)')
plt.title('Which features predict Check-In?')
plt.show()